<div style="font-size: 2em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Moyenne par quart d'heure + jour de la semaine <br>
RMSE public : 4.9670 ?<br>
RMSE perso : 5.5962
</div>
<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Importation des bibliothèques
</div>

In [1]:
from Donnees_et_RMSE import *

<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Modèle de prévision
</div>

In [2]:
def modele_prevision(train_modele, test_modele):
    # Initialisation -------------------------------------------------------------------------------------------
    # On convertit les indexe en colonnes classiques
    train = train_modele.reset_index()
    
    # On extrait tous les indexe station et commit_at_index sous forme de liste
    station_index = test_modele.index.get_level_values("station")
    commit_at_index = test_modele.index.get_level_values("commit_at")
    
    
    
    # Extraction des paramètres --------------------------------------------------------------------------------
    # On convertit les chaines de caractère en format date en on extrait les paramètres souhaités
    train["jour"] = train["commit_at"].dt.dayofweek
    train["heure"] = train["commit_at"].dt.hour
    train["minutes"] = train["commit_at"].dt.minute
    
    
      
    # Prédictions ----------------------------------------------------------------------------------------------
    # Moyenne par jour et quart d'heure de chaque station
    moyenne_station_jour = train.groupby(["station", "jour", "heure", "minutes"])["bikes"].mean().round().astype(int)
    
    # Moyennes de moins en moins précises en cas de valeurs manquantes
    moyenne_station_quart_heure = train.groupby(["station", "heure", "minutes"])["bikes"].mean().round().astype(int)
    moyenne_station_heure = train.groupby(["station", "heure"])["bikes"].mean().round().astype(int)
    moyenne_station = train.groupby("station")["bikes"].mean().round().astype(int)
    moyenne_globale = train["bikes"].mean().round().astype(int)
    
    # On procède pour chaque décalage individuellement
    for col, m in decalage.items():
        # Instant cible = commit_at + h minutes
        jour_cible = (commit_at_index + pd.Timedelta(minutes=m)).dayofweek
        heure_cible = (commit_at_index + pd.Timedelta(minutes=m)).hour
        minute_cible = (commit_at_index + pd.Timedelta(minutes=m)).minute
        # On relie chaque station aux paramètres
        cle = list(zip(station_index, jour_cible, heure_cible, minute_cible))
        # On associe chaque (station_index, paramètres) à la moyenne du nombre de vélos calculée
        valeurs = pd.Series(cle, index=test_modele.index).map(moyenne_station_jour)
        # En cas de valeur manquante, on utilise les moyennes calcuées avant
        cle = list(zip(station_index, heure_cible, minute_cible))
        valeurs = valeurs.fillna(pd.Series(cle, index=test_modele.index).map(moyenne_station_quart_heure))
        cle = list(zip(station_index, heure_cible))
        valeurs = valeurs.fillna(pd.Series(cle, index=test_modele.index).map(moyenne_station_heure))
        valeurs = valeurs.fillna(pd.Series(station_index, index=test_modele.index).map(moyenne_station))
        valeurs = valeurs.fillna(moyenne_globale)
        # On sauvegarde le résultat en tant qu'entier
        test_modele[col] = valeurs.astype(int)
    
    # On renvoie les prédictions
    return test_modele

<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Prédiction pour calculer le score RMSE perso
</div>
On prédit avec history_toulouse_apprentissage

In [3]:
history_toulouse_test = modele_prevision(history_toulouse_apprentissage, history_toulouse_test)

Calcul du score RMSE

In [4]:
rmse = rmse_calcul(history_toulouse_test, history_toulouse_validation)
print(f"RMSE : {rmse}")

RMSE : 5.596182989108002


<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
On recalcule tout avec TOUTES les données d'entraînement fournies
</div>

In [5]:
history_toulouse_test = modele_prevision(history_toulouse_train, history_toulouse_test)

<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Sauvegarde du résultat
</div>

In [6]:
history_toulouse_test.to_parquet("BA_prediction_4_quart_heure_semaine.parquet", index=True)
history_toulouse_test

,horizon,bikes_+15min,bikes_+30min,bikes_+45min,bikes_+60min,bikes_+75min,bikes_+90min,bikes_+105min,bikes_+120min
station,commit_at,,,,,,,,
00001 - POIDS DE L'HUILE,2026-07-31 21:45:00,6,5,5,4,4,4,3,3
00002 - LAFAYETTE,2026-07-31 23:45:00,6,6,5,5,5,5,4,4
00003 - ALSACE - LORRAINE - POMME,2025-11-10 11:45:00,11,10,11,11,9,9,8,8
00003 - ALSACE-LORRAINE - POMME,2026-07-31 21:45:00,6,6,5,5,4,3,3,2
00003 - POMME,2025-01-05 05:45:00,7,7,7,7,8,7,7,8
...,...,...,...,...,...,...,...,...,...
NARBONNE - CLOTASSES,2025-08-22 19:15:00,0,0,0,0,0,0,0,0
PASTEUR - LAVOISIER,2025-08-05 10:15:00,0,0,0,0,0,0,0,0
PLACE GEORGES BRASSENS,2025-08-05 11:00:00,0,0,0,0,0,0,0,0
